In [1]:
"""
exp13 Step 01: Stress Test Of The Frozen E1M1 On 2005-2014 (PROTOCOL.md §4, step 01)

STRESS = the first session where both the 200-session average and vix_level_prev exist (on or after 2005-10-04) ->
2014-12-31, the only span where E1M1 has never run. SPY bars and every VIX load are cut at
so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of 2015-2026 is read.

The frozen E1M1 (exp12's code), the unmodified E1 and E2M1 (reported alongside) and buy-and-hold, each as one continuous
path (start invested, no forced buy-back) over three windows: STRESS as a whole; 2007-10-01 -> 2009-08-12 (the 2008 bear
market); 2009-08-13 -> 2014-12-31 (exp12's exploration span: E1M1 +104.51% and E1 +49.00% are asserted). Then the full
episode scorecard of E1M1 over STRESS (VIX at the exit, maximum while out, at the buy-back; for every M1 re-entry the
lowest close until the next exit and the drawdown from the buy-back price to it).

E1M1 was chosen after the fact on 2015-2026; this step cannot make that out-of-sample evidence. Caveat: the 2008 bear
market and its VIX peak are known to everyone. Outputs: store04_experiments/exp13_e1m1_robustness/step01_stress_data/.
Trials: 2 (stage "exploration": E1M1, E2M1).
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp13_e1m1_robustness import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp13_e1m1_robustness.robustness import get_span_data_dict, check_last_date_dict, get_stress_start_date, get_year_period_pdf, get_frozen_rule_tuple, \
    replay_rule_dict, get_path_metric_dict, get_stress_scorecard_pdf, check_step_trial_dict
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# CHECK THE REAL DATA ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_stress_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp13_e1m1_robustness	Protocol:	1.0	Config Hash:	c2e293d595
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step01_stress_data/
ℹ️ No earlier 'exploration' entry of exp13_e1m1_robustness in the trial log (config hash c2e293d595).


In [4]:
# CALL FUNCTION TO READ SPY AND THE VIX UP TO THE STRESS CUTOFF (NOTHING OF 2015-2026 IS READ) AND BUILD THE FEATURES
cutoff_date_str = exp_config.STRESS_DATA_CUTOFF_DATE_STR
span_data_dict = get_span_data_dict(cutoff_date_str)
ohlcv_array_dict, daily_pdf, vix_feature_pdf = span_data_dict["ohlcv_array_dict"], span_data_dict["daily_pdf"], span_data_dict["vix_feature_pdf"]
# DISPLAY AND ASSERT THE LAST DATES LOADED
print(f"SPY sessions loaded:\t{len(daily_pdf):,}\tfrom {span_data_dict['first_spy_date']}")
last_date_str_dict = check_last_date_dict(span_data_dict["last_date_dict"], cutoff_date_str, CHECK_REAL_DATA_BOOL)
# DEFINE THE STRESS START AND THE THREE WINDOWS
stress_start_date = get_stress_start_date(daily_pdf, vix_feature_pdf)
window_dict = {"stress": (str(stress_start_date), exp_config.STRESS_END_DATE_STR), "bear_2007_2009": exp_config.BEAR_WINDOW_TUPLE,
               "exp12_exploration": exp_config.EXP12_EXPLORATION_WINDOW_TUPLE}
for window_str, (date1_str, date2_str) in window_dict.items():
    window_date_list = [d for d in ohlcv_array_dict["session_date_list"] if pd.Timestamp(date1_str).date() <= d <= pd.Timestamp(date2_str).date()]
    print(f"Window {window_str}:\t{window_date_list[0]} -> {window_date_list[-1]}\t({len(window_date_list):,} sessions)")
    if CHECK_REAL_DATA_BOOL and window_str != "stress":
        assert str(window_date_list[0]) == date1_str and str(window_date_list[-1]) == date2_str, f"❌ Window {window_str} differs"

SPY sessions loaded:	2,568	from 2005-01-03
Last date loaded:	spy	2015-03-18	(cutoff 2015-03-18)
Last date loaded:	vix_daily	2015-03-18	(cutoff 2015-03-18)
Last date loaded:	vix3m_daily	2015-03-18	(cutoff 2015-03-18)
Window stress:	2005-10-18 -> 2014-12-31	(2,316 sessions)
Window bear_2007_2009:	2007-10-01 -> 2009-08-12	(471 sessions)
Window exp12_exploration:	2009-08-13 -> 2014-12-31	(1,356 sessions)


In [5]:
"""
E1M1, the unmodified E1 and E2M1 (exp12's code), each one continuous path per window, against buy-and-hold over the same
sessions. The exp12 exploration window reproduces exp12 step 01 (asserted).
"""
# DEFINE THE PATHS
candidate_dict_list = [exp_config.FROZEN_CANDIDATE_DICT, exp_config.UNMODIFIED_E1_CANDIDATE_DICT, exp_config.E2M1_CANDIDATE_DICT]
# LISTS TO HOLD THE ROWS AND THE SIMULATIONS OF THE STRESS WINDOW
row_dict_list, stress_simulation_dict = [], {}
# ITERATE OVER THE WINDOWS
for window_str, (date1_str, date2_str) in window_dict.items():
    # SIMULATE BUY AND HOLD AND DEFINE THE YEARS AS PERIODS
    buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, date1_str, date2_str)
    year_period_pdf = get_year_period_pdf(ohlcv_array_dict["session_date_list"], date1_str, date2_str)
    row_dict_list.append({"window": window_str, "candidate_name": "buy_hold", "first_date": date1_str, "last_date": date2_str,
                          "total_return": buy_hold_dict["metric_dict"]["total_return"], "buy_hold_total_return": buy_hold_dict["metric_dict"]["total_return"],
                          "excess_total_return": 0.0, "sharpe_ratio": buy_hold_dict["metric_dict"]["sharpe_ratio"], "max_drawdown": buy_hold_dict["metric_dict"]["max_drawdown"],
                          "in_market_pct": 1.0, "exit_count": 0})
    # ITERATE OVER THE PATHS
    for candidate_dict in candidate_dict_list:
        simulation_dict = replay_rule_dict(ohlcv_array_dict, get_frozen_rule_tuple(daily_pdf, vix_feature_pdf, candidate_dict), date1_str, date2_str)
        row_dict_list.append({"window": window_str, "candidate_name": candidate_dict["candidate_name"], **get_path_metric_dict(simulation_dict, buy_hold_dict, year_period_pdf, ohlcv_array_dict)})
        if window_str == "stress":
            stress_simulation_dict[candidate_dict["candidate_name"]] = simulation_dict
stress_pdf = pd.DataFrame(row_dict_list)
# ASSERT THE REPRODUCTION OF exp12 STEP 01 (2009-08-13 -> 2014-12-31)
reproduction_pdf = stress_pdf[stress_pdf["window"] == "exp12_exploration"].set_index("candidate_name")
reproduction_dict = {name: round(100 * reproduction_pdf.loc[name, "total_return"], 2) for name in ["E1M1", "E1none", "buy_hold"]}
print(f"Reproduction of exp12 step 01:\t{reproduction_dict}")
if CHECK_REAL_DATA_BOOL:
    assert reproduction_dict["E1M1"] == exp_config.REPRODUCTION_EXPLORATION_E1M1_TOTAL_PCT, "❌ E1M1 differs from exp12 step 01"
    assert reproduction_dict["E1none"] == exp_config.REPRODUCTION_EXPLORATION_E1_TOTAL_PCT, "❌ E1 differs from exp12 step 01"
    assert reproduction_dict["buy_hold"] == exp_config.REPRODUCTION_EXPLORATION_BUY_HOLD_PCT, "❌ Buy-and-hold differs from exp12 step 01"
    print("Reproduction asserts:\tPASSED")
# PREVIEW DATAFRAME
stress_pdf[["window", "candidate_name", "first_date", "last_date", "total_return", "buy_hold_total_return", "excess_total_return", "annualized_log_excess",
            "log_excess_ci_low_6m", "log_excess_ci_high_6m", "log_excess_ci_low_1m", "log_excess_ci_high_1m", "in_market_pct", "exit_count",
            "sharpe_ratio", "max_drawdown", "product_s_over_r"]].round(4)

Reproduction of exp12 step 01:	{'E1M1': np.float64(104.51), 'E1none': np.float64(49.0), 'buy_hold': np.float64(104.57)}
Reproduction asserts:	PASSED


,window,candidate_name,first_date,last_date,total_return,buy_hold_total_return,excess_total_return,annualized_log_excess,log_excess_ci_low_6m,log_excess_ci_high_6m,log_excess_ci_low_1m,log_excess_ci_high_1m,in_market_pct,exit_count,sharpe_ratio,max_drawdown,product_s_over_r
0,stress,buy_hold,2005-10-18,2014-12-31,0.7309,0.7309,0.0000,NaN,NaN,NaN,NaN,NaN,1.0000,0,0.3926,-0.5645,NaN
1,stress,E1M1,2005-10-18,2014-12-31,0.5332,0.7309,-0.1977,-0.0131,-0.0418,0.0203,-0.0457,0.0245,0.9352,33,0.3347,-0.5713,0.8881
2,stress,E1none,2005-10-18,2014-12-31,0.5400,0.7309,-0.1909,-0.0126,-0.1095,0.1083,-0.0931,0.0756,0.7343,33,0.4763,-0.2325,0.8921
3,stress,E2M1,2005-10-18,2014-12-31,0.5826,0.7309,-0.1483,-0.0097,-0.0342,0.0152,-0.0344,0.0156,0.9637,13,0.3506,-0.5581,0.9152
4,bear_2007_2009,buy_hold,2007-10-01,2009-08-12,-0.3416,-0.3416,0.0000,NaN,NaN,NaN,NaN,NaN,1.0000,0,-0.4681,-0.5641,NaN
5,bear_2007_2009,E1M1,2007-10-01,2009-08-12,-0.3457,-0.3416,-0.0040,-0.0032,-0.0796,0.0678,-0.0707,0.0831,0.9106,6,-0.4913,-0.5438,0.9946
6,bear_2007_2009,E1none,2007-10-01,2009-08-12,-0.0565,-0.3416,0.2852,0.1878,-0.2732,0.6704,-0.1350,0.5324,0.1681,6,-0.3799,-0.1740,1.4348
7,bear_2007_2009,E2M1,2007-10-01,2009-08-12,-0.3318,-0.3416,0.0099,0.0078,-0.0679,0.0868,-0.0688,0.0816,0.9106,3,-0.4604,-0.5576,1.0154
8,exp12_exploration,buy_hold,2009-08-13,2014-12-31,1.0457,1.0457,0.0000,NaN,NaN,NaN,NaN,NaN,1.0000,0,0.9168,-0.1944,NaN
9,exp12_exploration,E1M1,2009-08-13,2014-12-31,1.0451,1.0457,-0.0007,-0.0001,-0.0376,0.0525,-0.0415,0.0487,0.9454,16,0.9876,-0.1405,1.0008


In [6]:
"""
Episode scorecard of E1M1 over STRESS. S / R > 1 = bought back lower than sold; vix_at_exit / vix_max_out /
vix_at_reentry = vix_level_prev at the exit decision, its maximum while out, at the buy-back decision; for an M1
re-entry, lowest_close_to_next_exit and drawdown_to_low_pct = SPY's lowest close until the next exit and the fall from
the buy-back price to it.
"""
# BUILD THE SCORECARDS
scorecard_pdf_dict = {name: get_stress_scorecard_pdf(simulation_dict, ohlcv_array_dict, daily_pdf, vix_feature_pdf) for name, simulation_dict in stress_simulation_dict.items()}
e1m1_scorecard_pdf = scorecard_pdf_dict["E1M1"]
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_max_out",
                      "vix_at_reentry", "lowest_close_to_next_exit", "drawdown_to_low_pct", "buy_hold_return_out", "max_decline_out_pct", "open_at_end"]
# DISPLAY THE SUMMARY OF EVERY PATH
for name, scorecard_pdf in scorecard_pdf_dict.items():
    reason_dict = scorecard_pdf["reentry_trigger"].value_counts().to_dict() if len(scorecard_pdf) else {}
    print(f"{name}:\t{len(scorecard_pdf)} episodes, {int((scorecard_pdf['s_over_r'] > 1).sum()) if len(scorecard_pdf) else 0} bought back lower\t"
          f"product of S/R {float(np.prod(scorecard_pdf['s_over_r'])) if len(scorecard_pdf) else 1.0:.4f}\treasons {reason_dict}")
# DISPLAY THE FULL SCORECARD OF E1M1
e1m1_scorecard_pdf[scorecard_col_list].round(4)

E1M1:	33 episodes, 8 bought back lower	product of S/R 0.8881	reasons {'original rule': 18, 'M1': 15}
E1none:	33 episodes, 1 bought back lower	product of S/R 0.8921	reasons {'original rule': 33}
E2M1:	13 episodes, 3 bought back lower	product of S/R 0.9152	reasons {'M1': 9, 'original rule': 4}


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_max_out,vix_at_reentry,lowest_close_to_next_exit,drawdown_to_low_pct,buy_hold_return_out,max_decline_out_pct,open_at_end
0,2005-10-18,117.90,2005-10-31,120.77,0.9762,9,original rule,14.67,16.28,14.19,NaN,NaN,0.0242,-0.0067,False
1,2006-06-07,126.01,2006-06-08,126.24,0.9982,1,original rule,17.39,17.81,17.81,NaN,NaN,0.0017,-0.0240,False
2,2006-06-09,125.55,2006-06-16,125.02,1.0042,5,M1,18.13,23.35,16.04,123.82,-0.0096,-0.0044,-0.0256,False
3,2006-07-12,125.91,2006-07-20,124.96,1.0076,6,M1,13.18,18.59,15.14,124.11,-0.0068,-0.0077,-0.0280,False
4,2006-07-27,126.34,2006-07-28,127.83,0.9883,1,original rule,14.55,15.26,15.26,NaN,NaN,0.0116,-0.0001,False
5,2006-08-09,126.71,2006-08-10,127.28,0.9955,1,original rule,15.39,15.39,15.28,NaN,NaN,0.0043,-0.0035,False
6,2006-08-11,126.84,2006-08-15,128.71,0.9855,2,original rule,14.52,14.52,14.30,NaN,NaN,0.0146,-0.0002,False
7,2007-08-03,143.40,2007-08-06,146.71,0.9774,1,original rule,21.43,25.31,25.31,NaN,NaN,0.0229,-0.0061,False
8,2007-08-14,143.05,2007-08-21,144.96,0.9868,5,M1,26.40,31.65,26.36,143.66,-0.0090,0.0132,-0.0424,False
9,2007-08-28,143.59,2007-08-29,146.66,0.9791,1,original rule,22.66,26.39,26.39,NaN,NaN,0.0212,-0.0001,False


In [7]:
# DISPLAY THE E1M1 EPISODES OF 2007-2009
print("E1M1 episodes with an exit in 2007-2009:")
e1m1_scorecard_pdf[[2007 <= d.year <= 2009 for d in e1m1_scorecard_pdf["exit_date"]]][scorecard_col_list].round(4) if len(e1m1_scorecard_pdf) else e1m1_scorecard_pdf

E1M1 episodes with an exit in 2007-2009:


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,reentry_trigger,vix_at_exit,vix_max_out,vix_at_reentry,lowest_close_to_next_exit,drawdown_to_low_pct,buy_hold_return_out,max_decline_out_pct,open_at_end
7,2007-08-03,143.40,2007-08-06,146.71,0.9774,1,original rule,21.43,25.31,25.31,NaN,NaN,0.0229,-0.0061,False
8,2007-08-14,143.05,2007-08-21,144.96,0.9868,5,M1,26.40,31.65,26.36,143.66,-0.0090,0.0132,-0.0424,False
9,2007-08-28,143.59,2007-08-29,146.66,0.9791,1,original rule,22.66,26.39,26.39,NaN,NaN,0.0212,-0.0001,False
10,2007-09-07,145.85,2007-09-11,147.60,0.9881,2,original rule,23.93,27.51,27.51,NaN,NaN,0.0119,-0.0105,False
11,2007-11-07,147.64,2007-11-14,147.18,1.0031,5,M1,21.43,30.76,23.98,140.80,-0.0433,-0.0033,-0.0252,False
12,2007-12-11,148.30,2007-12-12,149.14,0.9944,1,original rule,20.76,23.47,23.47,NaN,NaN,0.0055,-0.0076,False
13,2007-12-14,147.20,2007-12-21,148.27,0.9928,5,M1,22.45,24.61,20.66,147.52,-0.0051,0.0071,-0.0221,False
14,2007-12-27,147.39,2008-02-01,139.35,1.0577,24,M1,18.73,30.86,26.14,68.07,-0.5115,-0.0547,-0.1452,False
15,2009-06-22,89.35,2009-06-25,92.03,0.9709,3,original rule,27.79,31.22,29.16,NaN,NaN,0.0298,-0.0057,False
16,2009-07-07,88.09,2009-07-13,90.07,0.9780,4,original rule,29.19,30.94,28.98,NaN,NaN,0.0222,-0.0125,False


In [8]:
# SAVE THE RESULTS
write_csv_file_to_path(stress_pdf, f"{output_path_str}stress_path_data.csv", "W")
write_csv_file_to_path(pd.concat([scorecard_pdf.assign(candidate_name=name) for name, scorecard_pdf in scorecard_pdf_dict.items()], ignore_index=True),
                       f"{output_path_str}stress_scorecard_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step01_stress_data/stress_path_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step01_stress_data/stress_path_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step01_stress_data/stress_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp13_e1m1_robustness/step01_stress_data/stress_scorecard_data.csv' Has Been Saved!


In [9]:
# CHECK THE TRIAL BUDGET AND THE EXPECTED COUNT BEFORE LOGGING
trial_log_pdf = read_trial_log_pdf()
check_step_trial_dict(trial_log_pdf, "step01", 2)
# LOG E1M1 AND E2M1 AS EXPLORATION TRIALS (THEIR METRICS OVER THE THREE WINDOWS; WINDOW COLUMNS = STRESS)
logged_dict_list = []
for candidate_dict in [exp_config.FROZEN_CANDIDATE_DICT, exp_config.E2M1_CANDIDATE_DICT]:
    metric_dict = {f"{row['window']}_{key}": value for _, row in stress_pdf[stress_pdf["candidate_name"] == candidate_dict["candidate_name"]].iterrows()
                   for key, value in row.items() if key not in ["window", "candidate_name"]}
    metric_dict.update({f"{window_str}_buy_hold_total_return": float(stress_pdf[(stress_pdf["window"] == window_str) & (stress_pdf["candidate_name"] == "buy_hold")]["total_return"].iloc[0])
                        for window_str in window_dict})
    logged_dict_list.append(log_trial_dict(exp_config, "exploration", {"step": "step01", "candidate_name": candidate_dict["candidate_name"], "exit_rule": candidate_dict["exit_rule"],
                                                                       "modification": candidate_dict["modification"], "span": "stress"}, metric_dict, False,
                                           fold_row_in={"valid_start": window_dict["stress"][0], "valid_end": window_dict["stress"][1]},
                                           note_str_in="exp13 step01 stress test 2005-2014 (frozen exp12 rule, continuous replay)"))
print(f"Trials logged:\t{len(logged_dict_list)}\tfirst logged_ts {logged_dict_list[0]['logged_ts']}\ttest_window_touched: {sorted({row['test_window_touched'] for row in logged_dict_list})}")

Trials logged:	2	first logged_ts 2026-10-09T19:57:49	test_window_touched: [False]


In [10]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,multivariate_check,1,1,0,2026-10-06T18:27:48,2026-10-06T18:27:48
1,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
2,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
3,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
4,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
5,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
6,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
7,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
8,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
9,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
